# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [17]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")


✅ Libraries imported successfully!
Boto3 version: 1.37.3
Rasterio version: 1.4.3


In [18]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [19]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [20]:

EVENT_NAME = '202409_Hurricane_Helene'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'usda'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [21]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [22]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 216 .tif files in the S3 bucket.


['drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_20240914T233706_DVR_RTC20_G_gpuned_A5E3_VH.tif',
 'drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_20240914T233706_DVR_RTC20_G_gpuned_A5E3_VV.tif',
 'drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_20240914T233706_DVR_RTC20_G_gpuned_A5E3_WM.tif',
 'drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_20240914T233706_DVR_RTC20_G_gpuned_A5E3_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_20240914T233731_DVR_RTC20_G_gpuned_DF05_VH.tif',
 'drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_20240914T233731_DVR_RTC20_G_gpuned_DF05_VV.tif',
 'drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_20240914T233731_DVR_RTC20_G_gpuned_DF05_WM.tif',
 'drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_20240914T233731_DVR_RTC20_G_gpuned_DF05_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_20240914T233756_DVR_RTC20_G_gpuned_51DB_VH.tif',
 'drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_2024091

## Configure bucket and paths (no need to create session manually)

In [23]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [24]:
# Check current cache status using the imported function
check_cache_status()

📁 Cache directory does not exist: data_download/
   Creating cache directory...
✅ Cache directory created: data_download/


(0, 0)

In [25]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    _
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [26]:
keys


['drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_20240914T233706_DVR_RTC20_G_gpuned_A5E3_VH.tif',
 'drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_20240914T233706_DVR_RTC20_G_gpuned_A5E3_VV.tif',
 'drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_20240914T233706_DVR_RTC20_G_gpuned_A5E3_WM.tif',
 'drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_20240914T233706_DVR_RTC20_G_gpuned_A5E3_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_20240914T233731_DVR_RTC20_G_gpuned_DF05_VH.tif',
 'drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_20240914T233731_DVR_RTC20_G_gpuned_DF05_VV.tif',
 'drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_20240914T233731_DVR_RTC20_G_gpuned_DF05_WM.tif',
 'drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_20240914T233731_DVR_RTC20_G_gpuned_DF05_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_20240914T233756_DVR_RTC20_G_gpuned_51DB_VH.tif',
 'drcs_activations/202409_Hurricane_Helene/usda/S1A_IW_2024091

In [27]:
def create_cog_filename_sentinel1_usda(f, EVENT_NAME):
    """Create COG filename for Sentinel-1 USDA products with datetime at end."""
    from pathlib import Path
    import re
    
    filename = Path(f).stem
    extension = Path(f).suffix
    
    # Extract components from filename
    # Pattern: S1A_IW_20240914T233706_DVR_RTC20_G_gpuned_A5E3_VH
    pattern = r'(S1[AB])_IW_(\d{8})T(\d{6})_DVR_RTC(\d+)_.*_([A-Z0-9]+)_(VH|VV|WM|rgb)'
    match = re.search(pattern, filename)
    
    if match:
        satellite = match.group(1)
        date_str = match.group(2)
        time_str = match.group(3)
        resolution = match.group(4)
        proc_id = match.group(5)
        product_type = match.group(6)
        
        # Format datetime
        year = date_str[:4]
        month = date_str[4:6]
        day = date_str[6:8]
        hour = time_str[:2]
        minute = time_str[2:4]
        second = time_str[4:6]
        
        formatted_datetime = f"{year}-{month}-{day}T{hour}:{minute}:{second}Z"
        
        # Build new filename with datetime at end
        cog_filename = f'{EVENT_NAME}_{satellite}_USDA_RTC{resolution}_{product_type}_{proc_id}_{formatted_datetime}{extension}'
    else:
        # Fallback
        cog_filename = f'{EVENT_NAME}_{filename}{extension}'
    
    return cog_filename

filter_str = 'usda'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_sentinel1_usda(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")



Testing WM filename:
  202409_Hurricane_Helene_S1A_USDA_RTC20_VH_A5E3_2024-09-14T23:37:06Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_VV_A5E3_2024-09-14T23:37:06Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_WM_A5E3_2024-09-14T23:37:06Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_rgb_A5E3_2024-09-14T23:37:06Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_VH_DF05_2024-09-14T23:37:31Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_VV_DF05_2024-09-14T23:37:31Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_WM_DF05_2024-09-14T23:37:31Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_rgb_DF05_2024-09-14T23:37:31Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_VH_51DB_2024-09-14T23:37:56Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_VV_51DB_2024-09-14T23:37:56Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_WM_51DB_2024-09-14T23:37:56Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_rgb_51DB_2024-09-14T23:37:56Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_VH_8F21_2024-09-14T23:38:20Z.tif
  202409_Hurri

In [28]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_sentinel1_usda, 
                                target_dir = "Sentinel-1/USDA", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202409_Hurricane_Helene_S1A_USDA_RTC20_VH_6445_2024-09-21T23:30:25Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_VV_6445_2024-09-21T23:30:25Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_WM_6445_2024-09-21T23:30:25Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_rgb_6445_2024-09-21T23:30:25Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_VH_BBD7_2024-09-21T23:30:51Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_VV_BBD7_2024-09-21T23:30:51Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_WM_BBD7_2024-09-21T23:30:51Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_rgb_BBD7_2024-09-21T23:30:51Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_VH_462E_2024-09-21T23:31:16Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_VV_462E_2024-09-21T23:31:16Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_WM_462E_2024-09-21T23:31:16Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_rgb_462E_2024-09-21T23:31:16Z.tif
  202409_Hurricane_Helene_S1A_USDA_RTC20_VH_77F6_2024-09-21T23:31:41Z.tif
  202409_Hurrica

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0.0001230349444085732, max=2.8609976768493652, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [PREDICTOR] Data type: float32, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpll5k208v_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpj3ir5bcf.tif


...

   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=0, max=255, center sample non-zero=999993/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=0, max=255, center sample non-zero=999993/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=0, max=255, center sample non-zero=999993/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmp96crp2cu_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpy8okjy60.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/USDA/202409_Hurricane_Helene_S1A_USDA_RTC20_rgb_9E7F_2024-10-01T23:47:57Z.tif
   [MEMORY] Final: 3033.1 MB (Change: +34.7 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202409_Hurricane_Helene_S1A_USDA_RTC20_rgb_9E7F_2024-10-01T23:47:57Z.tif

✅ Batch processing complete: 116 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-1/USDA/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-1/USDA/files_converted.csv
📁 COGs saved locally to: output/202409_Hurricane_Helene

📊 BATCH PROCESSING SUMMARY
Total files processed: 116
Successful: 116
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-10T17:21:54.444098


## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [13]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")


📊 Memory Usage Summary:
  Current memory usage: 2220.6 MB
  Available memory: 122176.7 MB
  Memory percent used: 4.1%
